<a href="https://colab.research.google.com/github/rdelhibabu/Lindbladian_SCT/blob/main/Lindbladian_SCT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install required packages (uncomment if running in a fresh Colab environment)
# !pip install mne pyriemann torch scikit-learn

import numpy as np
import scipy.linalg as la
from scipy.integrate import solve_ivp
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt

# Set random seed for reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Define Pauli matrices for N=2 Hilbert Space
sigma_x = np.array([[0, 1], [1, 0]], dtype=complex)
sigma_y = np.array([[0, -1j], [1j, 0]], dtype=complex)
sigma_z = np.array([[1, 0], [0, -1]], dtype=complex)
identity = np.eye(2, dtype=complex)

In [2]:
def generate_synthetic_eeg_data(num_trials=400, channels=4, samples=256):
    """
    Generates synthetic EEG epochs and simulates cognitive fatigue over time.
    Returns:
        X: array of shape (num_trials, channels, samples)
        Y: array of binary labels (Target/Non-Target)
        fatigue_ratios: array of Theta/Alpha power ratios tracking fatigue
    """
    X = np.zeros((num_trials, channels, samples))
    Y = np.random.randint(0, 2, num_trials)
    fatigue_ratios = np.zeros(num_trials)

    time = np.linspace(0, 1, samples)

    for t in range(num_trials):
        # Simulate temporal progression (Fatigue increases over trials)
        progression = t / num_trials

        # Theta power increases, Alpha power decreases with time (fatigue)
        theta_amp = 1.0 + 2.0 * progression
        alpha_amp = 2.0 - 1.5 * progression

        fatigue_ratios[t] = theta_amp / alpha_amp

        for c in range(channels):
            # Base signal: Theta (6 Hz) + Alpha (10 Hz) + Noise
            theta_wave = theta_amp * np.sin(2 * np.pi * 6 * time)
            alpha_wave = alpha_amp * np.sin(2 * np.pi * 10 * time)
            noise = np.random.normal(0, 0.5, samples)

            signal = theta_wave + alpha_wave + noise

            # Inject ErrP anomaly if Y==1
            if Y[t] == 1:
                errp = 1.5 * np.exp(-((time - 0.3)**2) / 0.01) * np.sin(2 * np.pi * 4 * time)
                signal += errp

            X[t, c, :] = signal

    return X, Y, fatigue_ratios

print("Generating synthetic EEG dataset...")
X_data, Y_labels, fatigue_track = generate_synthetic_eeg_data(num_trials=800)
print(f"Dataset generated: {X_data.shape}, Fatigue ratio range: [{fatigue_track.min():.2f}, {fatigue_track.max():.2f}]")

Generating synthetic EEG dataset...
Dataset generated: (800, 4, 256), Fatigue ratio range: [0.50, 5.97]


In [3]:
def log_euclidean_mapping(Sigma, Sigma_ref):
    """Maps covariance matrix to tangent space using Log-Euclidean metric."""
    inv_sqrt_ref = la.inv(la.sqrtm(Sigma_ref))
    mapped = inv_sqrt_ref @ Sigma @ inv_sqrt_ref
    return la.logm(mapped)

def riemannian_to_hilbert(X_epoch, Sigma_ref, lam=0.1, N=2):
    """
    Algorithm 2: Maps EEG Covariance to a Valid Density Matrix
    """
    C, S = X_epoch.shape

    # 1. Spatial Covariance
    Sigma = np.cov(X_epoch)

    # 2. Tangent Space Projection
    T = log_euclidean_mapping(Sigma, Sigma_ref)

    # Extract feature vector (simplified for mapping to N=2)
    # In practice, this requires a specific reduction from C channels to N dimensions
    v = np.diag(T)[:N]

    # 3. Hermitian Complexification
    # Using a simplified skew-symmetric proxy for demonstration
    A = np.zeros((N, N))
    A[0, 1] = 1; A[1, 0] = -1

    T_reduced = np.diag(v)
    rho_tilde = la.expm(T_reduced + 1j * lam * A)

    # 4. Enforce Positivity
    rho_H = 0.5 * (rho_tilde + rho_tilde.conj().T)
    evals, evecs = la.eigh(rho_H)
    evals[evals < 0] = 0 # Rectify negative eigenvalues
    rho_pos = evecs @ np.diag(evals) @ evecs.conj().T

    # 5. Trace Normalization
    rho = rho_pos / np.trace(rho_pos)

    return rho

# Establish baseline reference covariance
Sigma_ref = np.cov(np.mean(X_data[:50], axis=0))

In [4]:
class LindbladianOQS:
    def __init__(self, H, L_jump, gamma_max=1.0, k=5.0, beta=1.5):
        self.H = H
        self.L = L_jump
        self.gamma_max = gamma_max
        self.k = k
        self.beta = beta

    def compute_gamma(self, theta_alpha_ratio):
        """Dynamic fatigue parameter via logistic growth."""
        return self.gamma_max / (1.0 + np.exp(-self.k * (theta_alpha_ratio - self.beta)))

    def dissipator(self, rho, gamma):
        """Computes the Lindbladian dissipator term."""
        L_rho_Ldag = self.L @ rho @ self.L.conj().T
        anti_comm = 0.5 * (self.L.conj().T @ self.L @ rho + rho @ self.L.conj().T @ self.L)
        return gamma * (L_rho_Ldag - anti_comm)

    def rk4_step(self, rho, gamma, dt):
        """Algorithm 3: RK4 Integration for the Master Equation."""
        def derivative(p):
            commutator = -1j * (self.H @ p - p @ self.H)
            return commutator + self.dissipator(p, gamma)

        k1 = derivative(rho)
        k2 = derivative(rho + 0.5 * dt * k1)
        k3 = derivative(rho + 0.5 * dt * k2)
        k4 = derivative(rho + dt * k3)

        rho_new = rho + (dt / 6.0) * (k1 + 2*k2 + 2*k3 + k4)
        return rho_new / np.trace(rho_new) # Re-normalize to prevent drift

    def project_measurement(self, rho, projector):
        """Lüders' Rule for state collapse and Born's rule for probability."""
        prob = np.real(np.trace(projector @ rho))
        rho_collapsed = projector @ rho @ projector
        tr = np.trace(rho_collapsed)
        if tr > 1e-10:
            rho_collapsed = rho_collapsed / tr
        return prob, rho_collapsed

In [5]:
class EEG_LSTM(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2):
        super(EEG_LSTM, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # x shape: (batch, seq_len, channels)
        out, _ = self.lstm(x)
        # Take the output of the last time step
        out = self.fc(out[:, -1, :])
        return self.sigmoid(out).squeeze()

def train_lstm(X_train, Y_train, epochs=15):
    # Reshape for LSTM: (batch, seq_len, input_size)
    X_t = torch.tensor(X_train, dtype=torch.float32).permute(0, 2, 1)
    Y_t = torch.tensor(Y_train, dtype=torch.float32)

    model = EEG_LSTM(input_size=X_train.shape[1])
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.005)

    model.train()
    for ep in range(epochs):
        optimizer.zero_grad()
        outputs = model(X_t)
        loss = criterion(outputs, Y_t)
        loss.backward()
        optimizer.step()
    return model

def predict_lstm(model, X_test):
    model.eval()
    X_t = torch.tensor(X_test, dtype=torch.float32).permute(0, 2, 1)
    with torch.no_grad():
        preds = model(X_t).numpy()
    return preds

In [6]:
# --- Setup OQS Operators ---
# Hamiltonian driving the cognitive processing
H_cognitive = 0.5 * sigma_x
# Dephasing Jump Operator (Decision Fatigue)
L_dephase = sigma_z

# Measurement Projectors for ErrP Detection
M_target = np.array([[1, 0], [0, 0]], dtype=complex)
M_nontarget = np.array([[0, 0], [0, 1]], dtype=complex)

oqs_model = LindbladianOQS(H_cognitive, L_dephase)

# --- Chronological Evaluation Parameters ---
num_blocks = 4
block_size = len(X_data) // num_blocks
dt = 0.1

results = {"LSTM_Acc": [], "CQS_Acc": [], "OQS_Acc": [], "OQS_KLD": []}

print("Starting Chronological Evaluation Pipeline...")

for block in range(num_blocks - 1):
    print(f"\n--- Training on Blocks 0 to {block}, Testing on Block {block+1} ---")

    # Train Data (Accumulating history)
    train_idx = (block + 1) * block_size
    X_train = X_data[:train_idx]
    Y_train = Y_labels[:train_idx]

    # Test Data (Next adjacent block to test fatigue generalizability)
    test_start = train_idx
    test_end = test_start + block_size
    X_test = X_data[test_start:test_end]
    Y_test = Y_labels[test_start:test_end]
    fatigue_test = fatigue_track[test_start:test_end]

    # 1. Train & Predict LSTM
    lstm_net = train_lstm(X_train, Y_train)
    lstm_probs = predict_lstm(lstm_net, X_test)
    lstm_preds = (lstm_probs > 0.5).astype(int)
    lstm_acc = accuracy_score(Y_test, lstm_preds)

    # 2. Evaluate Quantum Models (CQS and OQS)
    cqs_preds = []
    oqs_preds = []
    oqs_probs = []

    # Initialize Density Matrix from calibration reference
    rho_current_oqs = riemannian_to_hilbert(X_test[0], Sigma_ref)
    rho_current_cqs = rho_current_oqs.copy()

    for i in range(block_size):
        # Extract features and map to Hilbert space (Simulation of Algorithm 2 incoming data)
        rho_incoming = riemannian_to_hilbert(X_test[i], Sigma_ref)

        # Determine dynamic fatigue rate for this specific trial
        # CQS has NO fatigue dissipation (gamma = 0)
        gamma_oqs = oqs_model.compute_gamma(fatigue_test[i])

        # Evolve state between stimuli (Algorithm 3)
        rho_current_cqs = oqs_model.rk4_step(rho_current_cqs, gamma=0.0, dt=dt)
        rho_current_oqs = oqs_model.rk4_step(rho_current_oqs, gamma=gamma_oqs, dt=dt)

        # Measure/Classify
        prob_cqs, _ = oqs_model.project_measurement(rho_current_cqs, M_target)
        prob_oqs, rho_current_oqs = oqs_model.project_measurement(rho_current_oqs, M_target)

        cqs_preds.append(1 if prob_cqs > 0.5 else 0)
        oqs_preds.append(1 if prob_oqs > 0.5 else 0)
        oqs_probs.append(prob_oqs)

    cqs_acc = accuracy_score(Y_test, cqs_preds)
    oqs_acc = accuracy_score(Y_test, oqs_preds)

    # Compute KL Divergence for OQS
    # (Assuming empirical distribution is derived from true labels + smoothing)
    epsilon = 1e-5
    P_emp = np.clip(Y_test, epsilon, 1 - epsilon)
    P_model = np.clip(np.array(oqs_probs), epsilon, 1 - epsilon)
    kld = np.mean(P_emp * np.log(P_emp / P_model) + (1 - P_emp) * np.log((1 - P_emp) / (1 - P_model)))

    results["LSTM_Acc"].append(lstm_acc)
    results["CQS_Acc"].append(cqs_acc)
    results["OQS_Acc"].append(oqs_acc)
    results["OQS_KLD"].append(kld)

    print(f"LSTM Accuracy: {lstm_acc:.3f}")
    print(f"Closed Quantum (CQS) Accuracy: {cqs_acc:.3f}")
    print(f"Open Quantum (OQS) Accuracy: {oqs_acc:.3f} | KLD: {kld:.4f}")

# Final Report
print("\n=== Final Aggregate Results ===")
print(f"Mean LSTM Accuracy: {np.mean(results['LSTM_Acc']):.3f}")
print(f"Mean CQS Accuracy: {np.mean(results['CQS_Acc']):.3f}")
print(f"Mean Proposed OQS Accuracy: {np.mean(results['OQS_Acc']):.3f}")

Starting Chronological Evaluation Pipeline...

--- Training on Blocks 0 to 0, Testing on Block 1 ---
LSTM Accuracy: 0.500
Closed Quantum (CQS) Accuracy: 0.520
Open Quantum (OQS) Accuracy: 0.510 | KLD: 2.9481

--- Training on Blocks 0 to 1, Testing on Block 2 ---
LSTM Accuracy: 0.530
Closed Quantum (CQS) Accuracy: 0.510
Open Quantum (OQS) Accuracy: 0.550 | KLD: 2.6980

--- Training on Blocks 0 to 2, Testing on Block 3 ---


/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: RuntimeWarning: logm result may be inaccurate, approximate err = 3.819679763184832e-13
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: RuntimeWarning: logm result may be inaccurate, approximate err = 3.883276413873337e-13
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: RuntimeWarning: logm result may be inaccurate, approximate err = 4.120884306302449e-13
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: RuntimeWarning: logm result may be inaccurate, approximate err = 4.962791979360504e-13
  return f(*arrays, *other_args, **kwargs)


LSTM Accuracy: 0.515
Closed Quantum (CQS) Accuracy: 0.490
Open Quantum (OQS) Accuracy: 0.510 | KLD: 2.9428

=== Final Aggregate Results ===
Mean LSTM Accuracy: 0.515
Mean CQS Accuracy: 0.507
Mean Proposed OQS Accuracy: 0.523


/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: RuntimeWarning: logm result may be inaccurate, approximate err = 4.461465855354626e-13
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: RuntimeWarning: logm result may be inaccurate, approximate err = 4.976323370182593e-13
  return f(*arrays, *other_args, **kwargs)
